# Notebook 3: Recurring Motif Core Validation

This notebook compares `phase_b.pt` and `phase_c.pt` with the core motif-focused experiments:

1. `motif_families`: discover recurring multi-layer motifs directly in `z`
2. `motif_galleries`: inspect the strongest motifs by exemplars, purity, and cohesion
3. `motif_persistence`: measure where motifs appear, persist, and disappear across depth
4. `motif_predictiveness`: test whether motif membership predicts useful held-out structure
5. `motif_interventions`: test whether the strongest motifs produce member-specific causal effects

Phase C is better when it produces cleaner, more persistent, more predictive, or more causally specific recurring motifs than Phase B.


In [ ]:
# Colab-first setup: clone/update the repo, install it, and mount Drive.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Running outside Colab; using the current working tree.")


In [ ]:
from flow_circuits.data import build_cifar10_splits
from flow_circuits.evaluation.motif_validation import (
    CORE_MOTIF_EXPERIMENT_IDS,
    discover_motif_families,
    run_motif_gallery_experiment,
    run_motif_intervention_experiment,
    run_motif_persistence_experiment,
    run_motif_predictiveness_experiment,
)
from flow_circuits.training import load_components_from_checkpoint


## Config

Change the single `EXPERIMENTS` line below to run all five core motif experiments or any subset.
The notebook always compares `phase_b.pt` and `phase_c.pt`, never retrains, and caches every experiment under a notebook-local output directory.


In [ ]:
from pathlib import Path

RUN_MODE = "debug"  # "debug" or "full"
CONFIG_NAME = "resnet18_aligned"
EXPERIMENTS = "all"
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits")
EXPERIMENT_ROOT = DRIVE_ROOT / "experiments" / CONFIG_NAME
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb03_recurring_motif_core_validation" / CONFIG_NAME

PHASE_B_CHECKPOINT = EXPERIMENT_ROOT / "phase_b.pt"
PHASE_C_CHECKPOINT = EXPERIMENT_ROOT / "phase_c.pt"


In [ ]:
import json
import time

import torch

try:
    import pandas as pd
except ImportError:
    pd = None

MODEL_ORDER = [("phase_b", "Phase B"), ("phase_c", "Phase C")]
EXPERIMENT_LABELS = {
    "motif_families": "Motif Families",
    "motif_galleries": "Motif Galleries",
    "motif_persistence": "Motif Persistence",
    "motif_predictiveness": "Motif Predictiveness",
    "motif_interventions": "Motif Interventions",
}

if EXPERIMENTS == "all":
    SELECTED_EXPERIMENTS = list(CORE_MOTIF_EXPERIMENT_IDS)
else:
    SELECTED_EXPERIMENTS = [str(name) for name in EXPERIMENTS]
invalid = sorted(set(SELECTED_EXPERIMENTS) - set(CORE_MOTIF_EXPERIMENT_IDS))
if invalid:
    raise ValueError(f"Unknown experiments: {invalid}. Valid ids: {CORE_MOTIF_EXPERIMENT_IDS}")

if not PHASE_B_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_B_CHECKPOINT}")
if not PHASE_C_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_C_CHECKPOINT}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR = OUTPUT_DIR / "comparisons"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS_BY_MODE = {
    "debug": {
        "motif_families": {"max_images": 1000, "nodes_per_layer": 2, "bootstrap_iterations": 2},
        "motif_galleries": {"max_images": 1000, "topk": 4, "exemplar_count": 9},
        "motif_persistence": {},
        "motif_predictiveness": {"max_images": 1000, "topk": 4},
        "motif_interventions": {"max_images": 1000, "topk": 2, "min_image_set_size": 25, "alpha": 0.05},
    },
    "full": {
        "motif_families": {"max_images": 5000, "nodes_per_layer": 4, "bootstrap_iterations": 5},
        "motif_galleries": {"max_images": 5000, "topk": 8, "exemplar_count": 9},
        "motif_persistence": {},
        "motif_predictiveness": {"max_images": 5000, "topk": 8},
        "motif_interventions": {"max_images": 5000, "topk": 5, "min_image_set_size": 25, "alpha": 0.05},
    },
}
if RUN_MODE not in SETTINGS_BY_MODE:
    raise ValueError(f"RUN_MODE must be one of {sorted(SETTINGS_BY_MODE)}")


def _format_seconds(seconds):
    if seconds is None:
        return "?"
    seconds = int(max(0, round(seconds)))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


def _progress_logger(**event):
    stamp = time.strftime("%H:%M:%S")
    label = "Phase C" if event["checkpoint_tag"] == "phase_c" else "Phase B"
    stage = event["stage"].replace("_", " ")
    total = event.get("total")
    counter = stage if total is None else f"{stage} {event['completed']}/{total}"
    elapsed = _format_seconds(event.get("elapsed_seconds"))
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    if eta is None:
        print(f"[{stamp}] {label} | {event['experiment']} | {counter} | elapsed {elapsed} | {message}", flush=True)
    else:
        print(f"[{stamp}] {label} | {event['experiment']} | {counter} | elapsed {elapsed} | ETA {_format_seconds(eta)} | {message}", flush=True)


def _cache_path(tag, experiment_id):
    return OUTPUT_DIR / tag / f"{experiment_id}.json"


def _load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _should_run(experiment_id):
    return experiment_id in SELECTED_EXPERIMENTS


def _write_summary_table(rows, *, title):
    print(title)
    if not rows:
        print("  No rows to display.")
        return
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"Run mode      : {RUN_MODE}")
print(f"Config        : {CONFIG_NAME}")
print(f"Experiments   : {SELECTED_EXPERIMENTS}")
print(f"Phase B ckpt  : {PHASE_B_CHECKPOINT}")
print(f"Phase C ckpt  : {PHASE_C_CHECKPOINT}")
print(f"Output dir    : {OUTPUT_DIR}")

components_by_tag = {
    "phase_b": load_components_from_checkpoint(PHASE_B_CHECKPOINT, device=device),
    "phase_c": load_components_from_checkpoint(PHASE_C_CHECKPOINT, device=device),
}
base_config = components_by_tag["phase_b"].config
loader_batch_size = int(base_config["data"]["batch_size"])
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=loader_batch_size,
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)

RESULTS = {}
MOTIF_ARTIFACTS = {}


def _run_or_load(tag, experiment_id, run_fn):
    cache_path = _cache_path(tag, experiment_id)
    if cache_path.exists() and not FORCE_RERUN:
        print(f"Loading cached {experiment_id} for {tag}: {cache_path}")
        return _load_json(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Running {experiment_id} for {tag}; cache -> {cache_path}")
    return run_fn(cache_path)


def _ensure_motif_artifact(tag):
    if tag in MOTIF_ARTIFACTS and not FORCE_RERUN:
        return MOTIF_ARTIFACTS[tag]
    settings = SETTINGS_BY_MODE[RUN_MODE]["motif_families"]
    shared_panel = None
    if tag == "phase_c":
        shared_panel = _ensure_motif_artifact("phase_b")["selected_node_panel"]
    artifact = _run_or_load(
        tag,
        "motif_families",
        lambda cache_path, tag=tag: discover_motif_families(
            components_by_tag[tag],
            loaders["discovery"],
            device=device,
            checkpoint_tag=tag,
            max_images=settings["max_images"],
            nodes_per_layer=settings["nodes_per_layer"],
            bootstrap_iterations=settings["bootstrap_iterations"],
            node_panel=shared_panel,
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
    MOTIF_ARTIFACTS[tag] = artifact
    return artifact


## Experiment 1: Motif Families

Discover recurring multi-layer motifs directly in `z` on the discovery split, using a checkpoint-independent `q`-dispersion node panel.


In [ ]:
if _should_run("motif_families") or any(_should_run(name) for name in CORE_MOTIF_EXPERIMENT_IDS[1:]):
    motif_family_results = {tag: _ensure_motif_artifact(tag) for tag, _ in MODEL_ORDER}
    RESULTS["motif_families"] = motif_family_results
else:
    print("Skipping motif_families.")


In [ ]:
if "motif_families" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["motif_families"][tag]["summary"]
        rows.append({
            "checkpoint": label,
            "n_motifs": summary["n_motifs"],
            "mean_motif_stability": summary["mean_motif_stability"],
            "mean_supporting_layers": summary["mean_supporting_layers"],
            "mean_motif_size": summary["mean_motif_size"],
        })
    _write_summary_table(rows, title="Motif family summary")


## Experiment 2: Motif Galleries

Inspect the strongest motifs with exemplar-image ids, class purity, and within-motif cohesion.


In [ ]:
if _should_run("motif_galleries"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["motif_galleries"]
    gallery_results = {}
    for tag, _label in MODEL_ORDER:
        motif_artifact = _ensure_motif_artifact(tag)
        gallery_results[tag] = _run_or_load(
            tag,
            "motif_galleries",
            lambda cache_path, tag=tag, motif_artifact=motif_artifact: run_motif_gallery_experiment(
                components_by_tag[tag],
                loaders["discovery"],
                device=device,
                checkpoint_tag=tag,
                motif_artifact=motif_artifact,
                max_images=settings["max_images"],
                topk=settings["topk"],
                exemplar_count=settings["exemplar_count"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["motif_galleries"] = gallery_results
else:
    print("Skipping motif_galleries.")


In [ ]:
if "motif_galleries" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["motif_galleries"][tag]["summary"]
        rows.append({
            "checkpoint": label,
            "n_ranked_motifs": summary["n_ranked_motifs"],
            "mean_class_purity": summary["mean_class_purity"],
            "mean_cohesion": summary["mean_cohesion"],
        })
    _write_summary_table(rows, title="Motif gallery summary")


## Experiment 3: Motif Persistence

Measure which layers motifs occupy, how far they span through depth, and how continuous their layer support is.


In [ ]:
if _should_run("motif_persistence"):
    persistence_results = {}
    for tag, _label in MODEL_ORDER:
        motif_artifact = _ensure_motif_artifact(tag)
        persistence_results[tag] = _run_or_load(
            tag,
            "motif_persistence",
            lambda cache_path, tag=tag, motif_artifact=motif_artifact: run_motif_persistence_experiment(
                motif_artifact,
                checkpoint_tag=tag,
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["motif_persistence"] = persistence_results
else:
    print("Skipping motif_persistence.")


In [ ]:
if "motif_persistence" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["motif_persistence"][tag]["summary"]
        rows.append({
            "checkpoint": label,
            "mean_depth_span": summary["mean_depth_span"],
            "fraction_spanning_ge_3_layers": summary["fraction_spanning_ge_3_layers"],
            "mean_persistence_score": summary["mean_persistence_score"],
        })
    _write_summary_table(rows, title="Motif persistence summary")


## Experiment 4: Motif Predictiveness

Test whether held-out motif membership predicts a dominant class or higher-confidence model state better than base rates alone.


In [ ]:
if _should_run("motif_predictiveness"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["motif_predictiveness"]
    predictiveness_results = {}
    for tag, _label in MODEL_ORDER:
        motif_artifact = _ensure_motif_artifact(tag)
        predictiveness_results[tag] = _run_or_load(
            tag,
            "motif_predictiveness",
            lambda cache_path, tag=tag, motif_artifact=motif_artifact: run_motif_predictiveness_experiment(
                components_by_tag[tag],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                motif_artifact=motif_artifact,
                max_images=settings["max_images"],
                topk=settings["topk"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["motif_predictiveness"] = predictiveness_results
else:
    print("Skipping motif_predictiveness.")


In [ ]:
if "motif_predictiveness" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["motif_predictiveness"][tag]["summary"]
        rows.append({
            "checkpoint": label,
            "n_tested_motifs": summary["n_tested_motifs"],
            "mean_precision": summary["mean_precision"],
            "mean_lift_over_base_rate": summary["mean_lift_over_base_rate"],
            "mean_member_margin_lift": summary["mean_member_margin_lift"],
        })
    _write_summary_table(rows, title="Motif predictiveness summary")


## Experiment 5: Motif Interventions

Run residual-patch interventions on the strongest motifs and compare member-specific effects against controls.


In [ ]:
if _should_run("motif_interventions"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["motif_interventions"]
    intervention_results = {}
    for tag, _label in MODEL_ORDER:
        motif_artifact = _ensure_motif_artifact(tag)
        intervention_results[tag] = _run_or_load(
            tag,
            "motif_interventions",
            lambda cache_path, tag=tag, motif_artifact=motif_artifact: run_motif_intervention_experiment(
                components_by_tag[tag],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                motif_artifact=motif_artifact,
                max_images=settings["max_images"],
                topk=settings["topk"],
                min_image_set_size=settings["min_image_set_size"],
                alpha=settings["alpha"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
    RESULTS["motif_interventions"] = intervention_results
else:
    print("Skipping motif_interventions.")


In [ ]:
if "motif_interventions" in RESULTS:
    rows = []
    for tag, label in MODEL_ORDER:
        summary = RESULTS["motif_interventions"][tag]["summary"]
        rows.append({
            "checkpoint": label,
            "member_specific_count": summary["member_specific_count"],
            "validated_count": summary["validated_count"],
            "mean_member_delta_margin": summary["mean_member_delta_margin"],
        })
    _write_summary_table(rows, title="Motif intervention summary")


## Final Comparison

This last cell collects the main summary metric from each selected experiment so you can compare Phase B vs Phase C at a glance.


In [ ]:
comparison_rows = []
metric_lookup = {
    "motif_families": "n_motifs",
    "motif_galleries": "mean_class_purity",
    "motif_persistence": "mean_depth_span",
    "motif_predictiveness": "mean_lift_over_base_rate",
    "motif_interventions": "member_specific_count",
}
for experiment_id, metric_name in metric_lookup.items():
    if experiment_id not in RESULTS:
        continue
    comparison_rows.append({
        "experiment": EXPERIMENT_LABELS[experiment_id],
        "phase_b": RESULTS[experiment_id]["phase_b"]["summary"].get(metric_name, 0.0),
        "phase_c": RESULTS[experiment_id]["phase_c"]["summary"].get(metric_name, 0.0),
        "metric": metric_name,
    })
_write_summary_table(comparison_rows, title="Recurring motif core validation summary")
(COMPARISON_DIR / "summary.json").write_text(json.dumps(comparison_rows, indent=2), encoding="utf-8")
